## Testing the DIHC Feature Manager Package

In [3]:
# %pip install ipywidgets tqdm
# jupyter nbextension enable --py widgetsnbextension
# !jupyter nbextension enable --py widgetsnbextension

### Sample EEG Data Test

#### Settings and Data Reading

#####  Importing the package
Load the package "DIHC_FeatureManager" which is in the same directory as this notebook (or your main python script/notebook)

In [4]:
# Importing necessary modules
import numpy as np
import pandas as pd
from DIHC_FeatureManager.DIHC_FeatureManager import *

##### Data loading
Reading sample data from the file "signal_data.csv" which is in the same directory as this notebook (or your main python script/notebook)

In [5]:
print(f'Data reading started...')
sample_df = pd.read_csv('./eeg_signal_data.csv')
print(f'Data reading completed...')
print(f"Data read from file: ")
sample_df

Data reading started...
Data reading completed...
Data read from file: 


,signal,label
0,-17.777778,0
1,0.195360,0
2,0.195360,0
3,0.586081,0
4,0.195360,0
...,...,...
921595,-33.797314,0
921596,-27.545788,0
921597,-17.777778,0
921598,-8.791209,0


##### Data inspection
Observing the column names, shape and other basic information of the data

In [6]:
print(f'Columns available in the dataframe: {sample_df.columns}')
print(f'Columns available in the dataframe: {sample_df.shape}')
print(f'Dataframe details: ')
sample_df.info()

Columns available in the dataframe: Index(['signal', 'label'], dtype='object')
Columns available in the dataframe: (921600, 2)
Dataframe details: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 921600 entries, 0 to 921599
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   signal  921600 non-null  float64
 1   label   921600 non-null  int64  
dtypes: float64(1), int64(1)
memory usage: 14.1 MB


##### Settings and partial sample data access
This data file contains 3 columns: "time", "signal", and "label".
Only the "signal" column that contains the (time series) data will be used for feature extraction.
For simplicity only the first few seconds of the signal and few segments are used for feature extraction.

In [7]:
# Preset examples:
# 1) Low-frequency short recording
# signal_frequency = 10      # Hz
# segment_length   = 5       # s
# segment_overlap  = 0       # s
# total_segments   = 40000

# 2) Low-frequency long recording
# signal_frequency = 32      # Hz
# segment_length   = 90 * 60 # s
# segment_overlap  = 0       # s
# total_segments   = 4

# Active configuration
signal_frequency = 256  # Hz
segment_length = 5  # s
segment_overlap = 0  # s
total_segments = 4

print(f"""Settings:
- Signal frequency: {signal_frequency} Hz
- Segment length: {segment_length} s
- Segment overlap: {segment_overlap} s
- Total segments: {total_segments}
""")


Settings:
- Signal frequency: 256 Hz
- Segment length: 5 s
- Segment overlap: 0 s
- Total segments: 4



In [8]:
print(f'Data sub-sampling started...')
# sample_data = np.array([52, 54, 6, 45, 14, 40, 42, 48, 52, 20, 28, 8, 63, 47, 23])

# sample_data = sample_df['signal'].values.tolist()
sample_data = sample_df.loc[:(total_segments*segment_length)*signal_frequency-1, 'signal'].values#.tolist()
# sample_data = sample_df.loc[:(total_segments*segment_length)*signal_frequency-1, 'signal'].values#.tolist()
# sample_data = sample_df.loc[:20*signal_frequency-1, 'signal'].values#.tolist()
# sample_data = sample_df.loc[:5100, 'signal'].values#.tolist()
# sample_data = sample_df.iloc[:20*256-1, 0:1].values#.tolist()
# print(len(sample_data))
# print(sample_data.shape, sample_data)
print(f'Data sub-sampling completed...')

print(f"Sample data shape: {sample_data.shape}")
sample_data

Data sub-sampling started...
Data sub-sampling completed...
Sample data shape: (5120,)


array([-17.77777778,   0.19536019,   0.19536019, ...,  -0.58608059,
         0.19536019,  -0.19536019])

#### Segmentation Test

##### Segmentation using `get_segments_for_data` function
- Create the object of the class "DIHC_FeatureManager" and call the method "get_segments_for_data" to extract features from the data.
- Use different parameters of the method "get_segments_for_data" to extract different number of segments.

In [9]:
print(f'Data segmentation started...')

feature_manager = DIHC_FeatureManager()
segmented_data_array = feature_manager.get_segments_for_data(sample_data, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
# segmented_data_array = feature_manager.get_segments_for_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# segmented_data_array = feature_manager.get_segments_for_data(sample_data, segment_length=5, segment_overlap=3, signal_frequency=signal_frequency)

print(f'Data segmentation completed...')

Data segmentation started...
Data segmentation started...
Generating segments #1 
Segmentation started...
Generating segment# 1 
Generating segments #2 
Generating segment# 2 
Generating segments #3 
Generating segment# 3 
Generating segments #4 
Generating segment# 4 
Generating segments #5 
Segmentation finished...
Finished segmentation of data...
Data segmentation completed...


##### Display the segmented data

In [10]:
print(f"Segmented data array shape: {segmented_data_array.shape}")
print(f"Extracted segments data: ")
segmented_data_array

Segmented data array shape: (4, 1280)
Extracted segments data: 


array([[-17.77777778,   0.19536019,   0.19536019, ..., -15.82417582,
        -15.43345543, -18.94993895],
       [-21.68498168, -18.94993895, -16.21489621, ...,  29.89010989,
         42.002442  ,  50.5982906 ],
       [ 59.97557998,  67.78998779,  74.43223443, ...,   0.58608059,
          3.32112332,   4.88400488],
       [  4.49328449,   2.93040293,   4.88400488, ...,  -0.58608059,
          0.19536019,  -0.19536019]])

#### Feature Extraction Test

##### Feature extraction using `extract_features_from_data` function
- Create the object of the class "DIHC_FeatureManager" and call the method "extract_features_from_data" to extract features from the data
- Use different parameters of the method "extract_features_from_data" to extract different features
- Additionally can remove some computationally expensive features to save time and/or solve the memory exhaustion problem for larger segments

###### Select features and/or remove computationally expensive features

In [11]:
# Comment of uncommenting the following line to remove computationally expensive features

feature_names = None  ###For all features- None works similarly as [DIHC_FeatureGroup.all]
# feature_names = [DIHC_FeatureGroup.tdNl, DIHC_FeatureGroup.fdLin]  ###For some specific features- in this case, Time-domain non-linear and frequency domain linear features
# feature_names = DIHC_FeatureGroup.remove_computationally_expensive_features( comp_exp_list_index=4 ) ###For all features, except the level-4 computationally expensive ones
# feature_names = DIHC_FeatureGroup.remove_computationally_expensive_features( feature_list=[DIHC_FeatureGroup.tdNl, DIHC_FeatureGroup.fdLin], comp_exp_list_index=4 ) ###For some specific features- in this case, Time-domain non-linear and frequency domain linear features, except the level-4 computationally expensive ones

# # sel_feats = ['approximateEntropy', 'sampleEntropy', 'shannonEntropy', 'lempelZivComplexity', 'fd_maximum_delta', 'fd_mean_alpha']
# sel_feats = ['approximateEntropy', 'sampleEntropy', 'shannonEntropy', 'lempelZivComplexity']
# feature_names = DIHC_FeatureGroup.select_some_specific_features(sel_feature_list=sel_feats)

print(f"Final feature list: {feature_names}")

Final feature list: None


In [12]:
print(f'Feature extraction started...')

feature_manager = DIHC_FeatureManager()
feature_df = pd.DataFrame()

if feature_names is None:
    feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
else:
    feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=feature_names, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=5, segment_overlap=4, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.fdNlPw, DIHC_FeatureGroup.fdNlPwBnd], segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.tdNlEn, DIHC_FeatureGroup.td], segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.tdNlEn, DIHC_FeatureGroup.tdNl], segment_length=5, signal_frequency=signal_frequency)

print(f'Feature extraction completed...')

Feature extraction started...
Data started segmenting for features: []
Feature extraction started...
Extracting features for segment# 1
Segmentation started...
Generating segment# 1 
Feature extraction started...
For segment: 1, extracting feature: maximum 
For segment: 1, extracting feature: minimum 
For segment: 1, extracting feature: mean 
For segment: 1, extracting feature: median 
For segment: 1, extracting feature: standardDeviation 
For segment: 1, extracting feature: variance 
For segment: 1, extracting feature: kurtosis 
For segment: 1, extracting feature: skewness 
For segment: 1, extracting feature: numberOfZeroCrossing 
For segment: 1, extracting feature: positiveToNegativeSampleRatio 
For segment: 1, extracting feature: positiveToNegativePeakRatio 
For segment: 1, extracting feature: meanAbsoluteValue 
For segment: 1, extracting feature: approximateEntropy 
For segment: 1, extracting feature: sampleEntropy 
For segment: 1, extracting feature: permutationEntropy 
For segmen

##### Display all features

In [13]:
print(f"For a total of {feature_df.shape[0]} segments, {feature_df.shape[1]} features were extracted")
print(f"The name of the features are: {feature_df.columns}")
print(f"Extracted features: ")
feature_df


For a total of 4 segments, 92 features were extracted
The name of the features are: Index(['maximum', 'minimum', 'mean', 'median', 'standardDeviation', 'variance',
       'kurtosis', 'skewness', 'numberOfZeroCrossing',
       'positiveToNegativeSampleRatio', 'positiveToNegativePeakRatio',
       'meanAbsoluteValue', 'approximateEntropy', 'sampleEntropy',
       'permutationEntropy', 'singularValueDecompositionEntropy',
       'fuzzyEntropy', 'distributionEntropy', 'shannonEntropy', 'renyiEntropy',
       'lempelZivComplexity', 'hjorthMobility', 'hjorthComplexity',
       'fisherInfo', 'petrosianFd', 'katzFd', 'higuchiFd',
       'detrendedFluctuation', 'entropyProfiled_total_sampleEntropy',
       'entropyProfiled_average_sampleEntropy',
       'entropyProfiled_maximum_sampleEntropy',
       'entropyProfiled_minimum_sampleEntropy',
       'entropyProfiled_median_sampleEntropy',
       'entropyProfiled_standardDeviation_sampleEntropy',
       'entropyProfiled_variance_sampleEntropy',
  

,maximum,minimum,mean,median,standardDeviation,variance,kurtosis,skewness,numberOfZeroCrossing,positiveToNegativeSampleRatio,...,fd_variance_gamma,fd_kurtosis_gamma,fd_skewness_gamma,spectralEntropy,fd_bandPower,fd_bandPower_alpha,fd_bandPower_beta,fd_bandPower_delta,fd_bandPower_theta,fd_bandPower_gamma
0,67.79,-88.11,0.44,0.20,27.23,741.72,0.14,-0.33,86.0,1.04,...,1999673.39,4.19,1.96,3.59,763.78,280.09,17.83,388.16,72.30,7.97
1,92.01,-81.86,2.73,2.74,26.46,700.24,0.18,0.02,77.0,1.16,...,1119851.09,3.99,1.79,3.69,8.96,150.45,16.45,393.89,91.86,8.96
2,96.70,-133.43,-3.39,-2.54,35.90,1288.69,0.90,-0.08,82.0,0.89,...,2856644.69,4.43,1.80,3.57,7.78,334.31,26.79,731.57,158.11,7.78
3,62.32,-66.62,2.58,2.93,23.42,548.52,0.20,-0.23,107.0,1.35,...,1255472.71,5.98,2.23,3.80,8.69,160.13,19.87,286.76,64.70,8.69


##### Save all features

In [14]:
# # save_file_path = './eeg_all_features_matlab.csv'
# save_file_path = './eeg_all_features_python.csv'
# feature_df.to_csv(save_file_path, index=False)
# print(f"All EEG features successfully saved to: {save_file_path}")

#### Entropy (SampEn) Profile Test

##### Sample Entropy (SampEn) Profile extraction
- Create the object of the class "DIHC_FeatureManager" and call the method "extract_sampEn_profile_from_data" to extract Sample entropy (SampEn) profile from the data
- Use different parameters of the method "extract_sampEn_profile_from_data" to extract entropy profile for Sample entropy (SampEn)

In [15]:
print(f'Entropy profile extraction started...')

feature_manager = DIHC_FeatureManager()

sampEn_Profile_df = feature_manager.extract_sampEn_profile_from_data(sample_data, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
# sampEn_Profile_df = feature_manager.extract_sampEn_profile_from_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# sampEn_Profile_df = feature_manager.extract_sampEn_profile_from_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# sampEn_Profile_df = feature_manager.extract_sampEn_profile_from_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# sampEn_Profile_df = feature_manager.extract_sampEn_profile_from_data(sample_data, segment_length=5, segment_overlap=0, signal_frequency=signal_frequency)

print(f'Entropy profile extraction completed...')

Entropy profile extraction started...
Entropy profile calculation started...
Generating SampEn entropy profile for segment# 1 
Segmentation started...
Generating segment# 1 
Generating SampEn entropy profile for segment# 2 
Generating segment# 2 
Generating SampEn entropy profile for segment# 3 
Generating segment# 3 
Generating SampEn entropy profile for segment# 4 
Generating segment# 4 
Generating SampEn entropy profile for segment# 5 
Segmentation finished...
Finished generating entropy profile for all segments...
Entropy profile extraction completed...


##### Display entropy profile data

In [16]:
print(f"SampEn entropy profile shape: {sampEn_Profile_df.shape}")
print(f"SampEn entropy profile values: ")
sampEn_Profile_df

SampEn entropy profile shape: (1722, 2)
SampEn entropy profile values: 


,Segment_No,sampEn_profile
0,0,3.595747
1,0,2.218863
2,0,1.868785
3,0,1.579396
4,0,1.368360
...,...,...
1717,3,0.000006
1718,3,0.000004
1719,3,0.000002
1720,3,0.000001


##### Save entropy profile data

In [17]:
# # save_file_path = './eeg_sampEn_profile_data_matlab.csv'
# save_file_path = './eeg_sampEn_profile_data_python.csv'
# sampEn_Profile_df.to_csv(save_file_path, index=False)
# print(f"SamEn EEG profile data successfully saved to: {save_file_path}")

### Sample Hypnogram Data Test

#### Settings and Data Reading

#####  Importing the package
Load the package "DIHC_FeatureManager" which is in the same directory as this notebook (or your main python script/notebook)

In [18]:
# Importing necessary modules
import numpy as np
import pandas as pd
from DIHC_FeatureManager.DIHC_FeatureManager import *

##### Data loading
Reading sample data from the file "signal_data.csv" which is in the same directory as this notebook (or your main python script/notebook)

In [19]:
print(f'Data reading started...')
sample_df = pd.read_csv('./hypnogram_signal_data.csv')
print(f'Data reading completed...')
print(f"Data read from file: ")
sample_df

Data reading started...
Data reading completed...
Data read from file: 


,signal,label
0,1,1
1,4,1
2,4,1
3,4,1
4,4,1
...,...,...
1867,2,0
1868,2,0
1869,2,0
1870,2,0


##### Data inspection
Observing the column names, shape and other basic information of the data

In [20]:
print(f'Columns available in the dataframe: {sample_df.columns}')
print(f'Columns available in the dataframe: {sample_df.shape}')
print(f'Dataframe details: ')
sample_df.info()

Columns available in the dataframe: Index(['signal', 'label'], dtype='object')
Columns available in the dataframe: (1872, 2)
Dataframe details: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1872 entries, 0 to 1871
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   signal  1872 non-null   int64
 1   label   1872 non-null   int64
dtypes: int64(2)
memory usage: 29.4 KB


##### Settings and partial sample data access
This data file contains 3 columns: "time", "signal", and "label".
Only the "signal" column that contains the (time series) data will be used for feature extraction.
For simplicity only the first few seconds of the signal and few segments are used for feature extraction.

In [21]:
signal_frequency = None
pseudo_signal_frequency = 256  # It should not be useful for hypnogram data
epocho_length = 30  # seconds
sample_per_minute = 2  # epochs per minute
segment_length = 90  # minutes
segment_overlap = 0  # minutes
total_segments = 4

print(f"""Settings:
- Pseudo signal frequency: {pseudo_signal_frequency} Hz
- Epoch length: {epocho_length} s
- Epochs per minute: {sample_per_minute}
- Segment length: {segment_length} min
- Segment overlap: {segment_overlap} min
- Total segments: {total_segments}
""")


Settings:
- Pseudo signal frequency: 256 Hz
- Epoch length: 30 s
- Epochs per minute: 2
- Segment length: 90 min
- Segment overlap: 0 min
- Total segments: 4



In [22]:
print(f'Data sub-sampling started...')
# sample_data = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 2, 2, 4, 4, 4, 2, 2, 0, 0])

# sample_data = sample_df['signal'].values.tolist()
sample_data = sample_df.loc[:total_segments*(segment_length*sample_per_minute)-1, 'signal'].values#.tolist()

# print(len(sample_data))
# print(sample_data.shape, sample_data)
print(f'Data sub-sampling completed...')

print(f"Sample data shape: {sample_data.shape}")
sample_data

Data sub-sampling started...
Data sub-sampling completed...
Sample data shape: (720,)


array([1, 4, 4, 4, 4, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 4, 4, 1, 0, 0, 0, 0, 0, 0,
       2, 1, 0, 1, 0, 1, 2, 1, 0, 1, 0, 0, 1, 1, 0, 2, 1, 2, 1, 0, 0, 0,
       0, 2, 1, 1, 1, 4, 4, 4, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2,
       0, 0, 1, 1, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 0, 0, 0, 0,
       0, 0, 1, 0, 2, 1, 2, 0, 0, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 1, 4, 4,
       4, 4, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 2, 1, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2, 1, 2, 0, 1,
       1, 1, 1, 0, 0, 0, 0, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 1, 0, 0, 0, 0, 2, 0, 0, 0,
       0, 1, 2, 1, 2, 0, 1, 2, 2, 1, 1, 1, 1, 0, 0,

#### Feature Extraction (Hypnogram) Test

##### Feature extraction using `extract_features_from_data` function
- Create the object of the class "DIHC_FeatureManager" and call the method "extract_features_from_data" to extract features from the data
- Use different parameters of the method "extract_features_from_data" to extract different features
- Additionally can remove some computationally expensive features to save time and/or solve the memory exhaustion problem for larger segments
- Additionally can also select some specific features (in this case, only the features that are related to hypnogram data are selected)

###### Select features and/or remove computationally expensive features

In [23]:
from DIHC_FeatureManager.DIHC_FeatureManager import DIHC_FeatureGroup


# Comment of uncommenting the following line to remove computationally expensive features

feature_names = None  ###For all features- None works similarly as [DIHC_FeatureGroup.all]
# feature_names = [DIHC_FeatureGroup.tdNl, DIHC_FeatureGroup.fdLin]  ###For some specific features- in this case, Time-domain non-linear and frequency domain linear features
# feature_names = DIHC_FeatureGroup.remove_computationally_expensive_features( comp_exp_list_index=4 ) ###For all features, except the level-4 computationally expensive ones
# feature_names = DIHC_FeatureGroup.remove_computationally_expensive_features( feature_list=[DIHC_FeatureGroup.tdNl, DIHC_FeatureGroup.fdLin], comp_exp_list_index=4 ) ###For some specific features- in this case, Time-domain non-linear and frequency domain linear features, except the level-4 computationally expensive ones

# # sel_feats = ['approximateEntropy', 'sampleEntropy', 'shannonEntropy', 'lempelZivComplexity', 'fd_maximum_delta', 'fd_mean_alpha']
sel_feats = ['approximateEntropy', 'sampleEntropy', 'shannonEntropy', 'lempelZivComplexity']
rem_feats_pattern = ['fd_', 'entropyProfiled_', 'spectral', 'positiveToNegative']
# feature_names = DIHC_FeatureGroup.select_some_specific_features(sel_feature_list=sel_feats, rem_feature_list=rem_feats_pattern)
# feature_names = DIHC_FeatureGroup.select_some_specific_features(sel_feature_list=sel_feats, rem_feature_list=None)
feature_names = DIHC_FeatureGroup.select_some_specific_features(sel_feature_list=None, rem_feature_list=rem_feats_pattern)
# feature_names = DIHC_FeatureGroup.select_some_specific_features(sel_feature_list=None, rem_feature_list=None)

print(f"Final feature list: {feature_names}")

Final feature list: [<DIHC_FeatureGroup.all: ['maximum', 'minimum', 'mean', 'median', 'standardDeviation', 'variance', 'kurtosis', 'skewness', 'numberOfZeroCrossing', 'meanAbsoluteValue', 'approximateEntropy', 'sampleEntropy', 'permutationEntropy', 'singularValueDecompositionEntropy', 'fuzzyEntropy', 'distributionEntropy', 'shannonEntropy', 'renyiEntropy', 'lempelZivComplexity', 'hjorthMobility', 'hjorthComplexity', 'fisherInfo', 'petrosianFd', 'katzFd', 'higuchiFd', 'detrendedFluctuation']>]


In [24]:
print(f'Feature extraction started...')

feature_manager = DIHC_FeatureManager()
feature_df = pd.DataFrame()

if feature_names is None:
    feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=(segment_length*sample_per_minute), segment_overlap=(segment_overlap*sample_per_minute), signal_frequency=signal_frequency)
else:
    feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=feature_names, segment_length=(segment_length*sample_per_minute), segment_overlap=(segment_overlap*sample_per_minute), signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=segment_length, segment_overlap=segment_overlap, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, segment_length=5, segment_overlap=4, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.fdNlPw, DIHC_FeatureGroup.fdNlPwBnd], segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.tdNlEn, DIHC_FeatureGroup.td], segment_length=5, signal_frequency=signal_frequency)
# feature_df = feature_manager.extract_features_from_data(sample_data, feature_names=[DIHC_FeatureGroup.tdNlEn, DIHC_FeatureGroup.tdNl], segment_length=5, signal_frequency=signal_frequency)

print(f'Feature extraction completed...')

Feature extraction started...
Data started segmenting for features: [<DIHC_FeatureGroup.all: ['maximum', 'minimum', 'mean', 'median', 'standardDeviation', 'variance', 'kurtosis', 'skewness', 'numberOfZeroCrossing', 'meanAbsoluteValue', 'approximateEntropy', 'sampleEntropy', 'permutationEntropy', 'singularValueDecompositionEntropy', 'fuzzyEntropy', 'distributionEntropy', 'shannonEntropy', 'renyiEntropy', 'lempelZivComplexity', 'hjorthMobility', 'hjorthComplexity', 'fisherInfo', 'petrosianFd', 'katzFd', 'higuchiFd', 'detrendedFluctuation']>]
Feature extraction started...
Extracting features for segment# 1
Segmentation started...
Generating segment# 1 
Feature extraction started...
For segment: 1, extracting feature: maximum 
For segment: 1, extracting feature: minimum 
For segment: 1, extracting feature: mean 
For segment: 1, extracting feature: median 
For segment: 1, extracting feature: standardDeviation 
For segment: 1, extracting feature: variance 
For segment: 1, extracting feature:

##### Display all features

In [25]:
print(f"For a total of {feature_df.shape[0]} segments, {feature_df.shape[1]} features were extracted")
print(f"The name of the features are: {feature_df.columns}")
print(f"Extracted features: ")
feature_df


For a total of 1 segments, 26 features were extracted
The name of the features are: Index(['maximum', 'minimum', 'mean', 'median', 'standardDeviation', 'variance',
       'kurtosis', 'skewness', 'numberOfZeroCrossing', 'meanAbsoluteValue',
       'approximateEntropy', 'sampleEntropy', 'permutationEntropy',
       'singularValueDecompositionEntropy', 'fuzzyEntropy',
       'distributionEntropy', 'shannonEntropy', 'renyiEntropy',
       'lempelZivComplexity', 'hjorthMobility', 'hjorthComplexity',
       'fisherInfo', 'petrosianFd', 'katzFd', 'higuchiFd',
       'detrendedFluctuation'],
      dtype='object')
Extracted features: 


,maximum,minimum,mean,median,standardDeviation,variance,kurtosis,skewness,numberOfZeroCrossing,meanAbsoluteValue,...,shannonEntropy,renyiEntropy,lempelZivComplexity,hjorthMobility,hjorthComplexity,fisherInfo,petrosianFd,katzFd,higuchiFd,detrendedFluctuation
0,4.0,0.0,0.77,0.0,1.02,1.04,1.6,1.39,0.0,0.77,...,1.67,-10.2,96.0,0.87,1.83,0.58,1.02,3.7,1.79,0.93


##### Save all features

In [26]:
# # save_file_path = './hyp_all_features_matlab.csv'
# save_file_path = './hyp_all_features_python.csv'
# feature_df.to_csv(save_file_path, index=False)
# print(f"All hypnogram features successfully saved to: {save_file_path}")